# N25 — Pace Agent

The Pace Agent answers one question: **how fast are we going this lap?**

It wraps the N06 XGBoost delta-lap-time model into a clean agent interface. Given the current race state (tyre compound, tyre life, lap number, track conditions, team/driver), it returns a lap time prediction with a confidence interval derived from bootstrap quantiles.

This is the first of seven sub-agents in the multi-agent strategy system (N25–N31). Its output feeds directly into the Strategy Orchestrator (N31) as the pace signal used to evaluate undercut/overcut windows and stint extension decisions.

## Pipeline position

<pre>
race_state (lap_number, tyre_life, compound, team, ...)
    │
    ▼
N25 — Pace Agent
    │
    ├── lap_time_pred        predicted lap time (seconds)
    ├── delta_vs_median      delta vs session median pace (+ = slower)
    └── confidence_interval  [p10, p90] bootstrap interval
</pre>




## Models used
- **N06** — XGBoost delta lap time (`data/models/lap_time/`)

## Steps
- **Step 0** — Setup & model loading
- **Step 1** — Feature preparation (race state → model input)
- **Step 2** — Inference + bootstrap confidence interval
- **Step 3** — `run_pace_agent(lap_state) → PaceOutput`
- **Step 4** — Demo: full 2025 race replay (lap by lap)
- **Step 5** — Export schema & config

---

In [20]:
# ── Step 0: Setup & model loading ──────────────────────────────────────────
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / '.git').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import numpy as np
import pandas as pd
import xgboost as xgb

# ── Paths ───────────────────────────────────────────────────────────────────
MODELS_DIR  = repo_root / 'data' / 'models' / 'lap_time'
OUTPUTS_DIR = repo_root / 'notebooks' / 'agents' / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)






In [21]:
# ── Load N06 model + features ───────────────────────────────────────────────
def load_pace_model(models_dir=MODELS_DIR):
    features = json.loads((models_dir / 'xgb_laptime_delta_feature_names.json').read_text())
    model = xgb.XGBRegressor()
    model.load_model(models_dir / 'xgb_laptime_delta_final.json')
    return model, features

In [22]:
MODEL, FEATURES = load_pace_model()

print(f"Model loaded — {len(FEATURES)} features")
print(f"Features: {FEATURES}")

Model loaded — 25 features
Features: ['DriverNumber', 'LapNumber', 'Stint', 'TyreLife', 'FreshTyre', 'Position', 'CompoundID', 'TeamID', 'LapsSincePitStop', 'FuelLoad', 'Year', 'FuelEffect', 'Prev_LapTime', 'Prev_TyreLife', 'Prev_SpeedST', 'AirTemp', 'TrackTemp', 'Humidity', 'Rainfall', 'laps_remaining', 'Cluster', 'mean_sector_speed', 'Prev_DegradationRate', 'Prev_CumulativeDeg', 'Prev_DegAcceleration']


---

## Step 1 — Feature preparation

The N06 model expects 25 features. This step defines the encoding maps (compound, team, circuit cluster) and a `build_lap_state()` function that packs raw race state values into a single-row DataFrame ready for inference.

A few details worth noting:

- **`FreshTyre`** — binary flag, true when `tyre_life ≤ 1` (first flying lap on a new set)
- **`FuelEffect`** — approximated as `fuel_load × 0.03s/kg`, same convention used in N06
- **`laps_remaining`** — computed from `total_laps − lap_number`; used by the model to capture end-of-race pace changes
- **`mean_sector_speed`** — when not available directly, `Prev_SpeedST` is used as a proxy
- **Degradation features** (`Prev_DegradationRate`, `Prev_CumulativeDeg`, `Prev_DegAcceleration`) — default to 0.0 on the first lap of a stint; the demo in Step 4 will populate these from the actual stint history


In [23]:
# ── Step 1: Feature preparation (race state → model input) ─────────────────

PROCESSED_DIR    = repo_root / 'data' / 'processed'
CLUSTER_PARQUET  = PROCESSED_DIR / 'circuit_clustering' / 'circuit_clusters_k4_2025.parquet'
LAPS_FEATURED    = PROCESSED_DIR / 'laps_featured_2025.parquet'
FEATURE_MANIFEST = PROCESSED_DIR / 'feature_manifest_laptime.json'


In [24]:
# ── Load encoding maps from saved artifacts ─────────────────────────────────
def load_encoding_maps():
    manifest    = json.loads(FEATURE_MANIFEST.read_text())
    compound_id = manifest['categorical_encoding']['Compound']

    clusters_df = pd.read_parquet(CLUSTER_PARQUET)[['GP_Name', 'Cluster']]
    circuit_cluster = dict(zip(clusters_df['GP_Name'], clusters_df['Cluster'].astype(int)))

    laps = pd.read_parquet(LAPS_FEATURED, columns=['Team', 'TeamID']).dropna()
    team_id = (laps.drop_duplicates('Team')
                    .set_index('Team')['TeamID']
                    .astype(int)
                    .to_dict())

    return compound_id, circuit_cluster, team_id


In [25]:
COMPOUND_ID, CIRCUIT_CLUSTER, TEAM_ID = load_encoding_maps()

print("Compound encoding:", COMPOUND_ID)
print("\nTeam encoding:", TEAM_ID)
print("\nCircuit → Cluster:")
for gp, c in sorted(CIRCUIT_CLUSTER.items()):
    print(f"  {gp:<22} → {c}")

Compound encoding: {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 'INTERMEDIATE': 3, 'WET': 4}

Team encoding: {'Red Bull Racing': 1, 'Alpine': 6, 'Mercedes': 2, 'Aston Martin': 5, 'Ferrari': 3, 'Williams': 7, 'Kick Sauber': 0, 'Racing Bulls': 0, 'Haas F1 Team': 10, 'McLaren': 4}

Circuit → Cluster:
  Austin                 → 1
  Baku                   → 1
  Barcelona              → 1
  Budapest               → 1
  Imola                  → 1
  Jeddah                 → 1
  Las Vegas              → 0
  Lusail                 → 0
  Marina Bay             → 1
  Melbourne              → 0
  Mexico City            → 3
  Miami                  → 1
  Monaco                 → 3
  Montréal               → 2
  Monza                  → 1
  Sakhir                 → 0
  Shanghai               → 1
  Silverstone            → 0
  Spa-Francorchamps      → 0
  Spielberg              → 1
  Suzuka                 → 0
  São Paulo              → 0
  Yas Island             → 1
  Zandvoort              → 0


In [26]:
# ── build_lap_state ─────────────────────────────────────────────────────────
def build_lap_state(
    driver_number, lap_number, stint, tyre_life, compound,
    position, team, laps_since_pit, fuel_load, year,
    prev_lap_time, prev_tyre_life, prev_speed_st,
    air_temp, track_temp, humidity, rainfall,
    total_laps, gp_name,
    mean_sector_speed=None,
    prev_deg_rate=0.0, prev_cum_deg=0.0, prev_deg_accel=0.0,
):
    """Pack raw race state into a single-row DataFrame ready for MODEL.predict()."""
    compound_id    = COMPOUND_ID.get(compound, 1)
    team_id        = TEAM_ID.get(team, 0)
    cluster        = CIRCUIT_CLUSTER.get(gp_name, 1)
    fresh_tyre     = int(tyre_life <= 1)
    fuel_effect    = fuel_load * 0.03
    laps_remaining = max(0, total_laps - lap_number)
    mss            = mean_sector_speed if mean_sector_speed is not None else prev_speed_st

    row = {
        'DriverNumber':          driver_number,
        'LapNumber':             lap_number,
        'Stint':                 stint,
        'TyreLife':              tyre_life,
        'FreshTyre':             fresh_tyre,
        'Position':              position,
        'CompoundID':            compound_id,
        'TeamID':                team_id,
        'LapsSincePitStop':      laps_since_pit,
        'FuelLoad':              fuel_load,
        'Year':                  year,
        'FuelEffect':            fuel_effect,
        'Prev_LapTime':          prev_lap_time,
        'Prev_TyreLife':         prev_tyre_life,
        'Prev_SpeedST':          prev_speed_st,
        'AirTemp':               air_temp,
        'TrackTemp':             track_temp,
        'Humidity':              humidity,
        'Rainfall':              rainfall,
        'laps_remaining':        laps_remaining,
        'Cluster':               cluster,
        'mean_sector_speed':     mss,
        'Prev_DegradationRate':  prev_deg_rate,
        'Prev_CumulativeDeg':    prev_cum_deg,
        'Prev_DegAcceleration':  prev_deg_accel,
    }
    return pd.DataFrame([row])[FEATURES]

In [27]:
# ── Sanity check ─────────────────────────────────────────────────────────────
sample = build_lap_state(
    driver_number=44, lap_number=10, stint=1, tyre_life=10,
    compound='MEDIUM', position=3, team='Mercedes', laps_since_pit=10,
    fuel_load=90, year=2025, prev_lap_time=92.4, prev_tyre_life=9,
    prev_speed_st=310.0, air_temp=28.0, track_temp=42.0,
    humidity=40.0, rainfall=0.0, total_laps=57, gp_name='Barcelona',
)
print(sample.to_string())


   DriverNumber  LapNumber  Stint  TyreLife  FreshTyre  Position  CompoundID  TeamID  LapsSincePitStop  FuelLoad  Year  FuelEffect  Prev_LapTime  Prev_TyreLife  Prev_SpeedST  AirTemp  TrackTemp  Humidity  Rainfall  laps_remaining  Cluster  mean_sector_speed  Prev_DegradationRate  Prev_CumulativeDeg  Prev_DegAcceleration
0            44         10      1        10          0         3           1       2                10        90  2025         2.7          92.4              9         310.0     28.0       42.0      40.0       0.0              47        1              310.0                   0.0                 0.0                   0.0


---

## Step 2 — Inference + confidence interval

With the feature vector ready, this step handles the actual prediction and adds an uncertainty estimate around it.

**Single-point prediction** is straightforward — one call to `model.predict()` on the prepared DataFrame.

**Confidence interval** uses a lightweight bootstrap: the input row is perturbed N=200 times with small Gaussian noise (0.5% relative) on the continuous features most subject to real-world variability (lap time, speed trap, temperatures, tyre life). The P10/P90 of the resulting predictions gives a plausible range that reflects sensor noise and lap-to-lap variation — not formal model uncertainty, but good enough for strategy decisions.

**Delta vs median** contextualises the raw prediction: a lap time of 92.4s means little without knowing whether that's fast or slow for this compound at this circuit. The delta is computed against the historical median for the same GP/year/compound from `laps_featured_2025.parquet`.


In [28]:
# ── Step 2: Inference + bootstrap confidence interval ──────────────────────

N_BOOTSTRAP = 200

def predict_lap_time(lap_state_df, model=MODEL):
    """Single-point prediction. Returns predicted lap time in seconds."""
    return float(model.predict(lap_state_df)[0])


def bootstrap_confidence_interval(lap_state_df, model=MODEL, n=N_BOOTSTRAP, seed=42):
    """
    Estimate a [P10, P90] interval by resampling the feature vector with
    small Gaussian noise on continuous features (speed, temps, tyre life).
    This reflects sensor noise / lap-to-lap variability, not model uncertainty.
    """
    rng = np.random.default_rng(seed)
    noise_cols = ['Prev_LapTime', 'Prev_SpeedST', 'mean_sector_speed',
                  'AirTemp', 'TrackTemp', 'TyreLife']

    base = lap_state_df.values.copy().astype(float)
    col_idx = {c: lap_state_df.columns.get_loc(c) for c in noise_cols}

    preds = []
    for _ in range(n):
        row = base.copy()
        for col, idx in col_idx.items():
            sigma = base[0, idx] * 0.005   # 0.5% relative noise
            row[0, idx] += rng.normal(0, sigma)
        preds.append(float(model.predict(pd.DataFrame(row, columns=lap_state_df.columns))[0]))

    return float(np.percentile(preds, 10)), float(np.percentile(preds, 90))


def session_median_lap_time(gp_name, year, compound, laps_df):
    """Compute median representative lap time for a given GP/year/compound from laps_featured."""
    mask = (
        (laps_df['GP_Name'] == gp_name) &
        (laps_df['Year']    == year) &
        (laps_df['Compound'] == compound)
    )
    subset = laps_df.loc[mask, 'LapTime_s'].dropna()
    return float(subset.median()) if len(subset) > 0 else None

In [29]:
N_BOOTSTRAP = 200

def predict_lap_time(lap_state_df, model=MODEL):
    """Predicts absolute lap time: Prev_LapTime + model delta."""
    delta = float(model.predict(lap_state_df)[0])
    prev  = float(lap_state_df['Prev_LapTime'].iloc[0])
    return prev + delta


def bootstrap_confidence_interval(lap_state_df, model=MODEL, n=N_BOOTSTRAP, seed=42):
    """P10/P90 interval via bootstrap noise on continuous features (2% relative)."""
    rng = np.random.default_rng(seed)
    noise_cols = ['Prev_LapTime', 'Prev_SpeedST', 'mean_sector_speed',
                  'AirTemp', 'TrackTemp', 'TyreLife']

    base    = lap_state_df.values.copy().astype(float)
    col_idx = {c: lap_state_df.columns.get_loc(c) for c in noise_cols}

    preds = []
    for _ in range(n):
        row = base.copy()
        for col, idx in col_idx.items():
            sigma = abs(base[0, idx]) * 0.02   # 2% relative noise
            row[0, idx] += rng.normal(0, sigma)
        df_row = pd.DataFrame(row, columns=lap_state_df.columns)
        delta  = float(model.predict(df_row)[0])
        preds.append(float(df_row['Prev_LapTime'].iloc[0]) + delta)

    return float(np.percentile(preds, 10)), float(np.percentile(preds, 90))




In [31]:
# ── Sanity check ─────────────────────────────────────────────────────────────
pred     = predict_lap_time(sample)
p10, p90 = bootstrap_confidence_interval(sample)
median   = session_median_lap_time('Barcelona', 2025, 'MEDIUM', LAPS_REF)

delta_pred = pred - float(sample['Prev_LapTime'].iloc[0])

print(f"Predicted lap time : {pred:.3f}s")
print(f"Predicted delta    : {delta_pred:+.3f}s  (vs prev lap)")
print(f"Confidence interval: [{p10:.3f}s, {p90:.3f}s]")
print(f"Session median     : {median:.3f}s")
print(f"Delta vs median    : {pred - median:+.3f}s")


Predicted lap time : 92.803s
Predicted delta    : +0.403s  (vs prev lap)
Confidence interval: [90.106s, 95.278s]
Session median     : 80.959s
Delta vs median    : +11.845s


---